# Deliverable A — Calibrated Default Probability

**Goal:** For every applicant in `validation.csv` and `test.csv` (13,306 total), produce a calibrated probability of default (`predicted_pd`) along with a 90 % prediction interval.

## Pipeline overview

| Step | What | Output |
|------|------|--------|
| 1 | Load & split data | `train`, `val_all`, `val_labeled` |
| 2 | Integrity checks | Planted violation counts |
| 3 | Feature engineering | 47 features incl. MNAR indicators |
| 4 | Build matrices | numpy arrays for CatBoost |
| 5 | 10-fold CV *(diagnostic)* | OOF AUC — model quality estimate only |
| 6 | 10-fold ensemble final model | `val_cal_all`, `test_cal` — calibrated PD per loan |

## Known traps handled

| Trap | Mitigation |
|------|------------|
| Outcome leakage | `OUTCOME_COLS` excluded before any fit |
| `prior_decision` collider | Excluded from feature set |
| MNAR bank-feed nulls | `{col}__missing` indicators added for all 6 bank-feed columns |
| Reject inference | Acknowledged; partially mitigated via `prior_underwriter_score` + val calibration |
| Single-model variance | 10-member `CatBoostEnsemble` via stratified K-fold (different training subsets) |
| Probability calibration | Isotonic regression on held-out `val_labeled` (independent of training) |

In [1]:
import sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, 'code')   # make code/ importable from repo root

import numpy as np
import pandas as pd

# ── Module imports (all logic lives in code/) ─────────────────────────────────
from config      import SEED, DATA_DIR, SUB_DIR, N_FOLDS
from data        import load_raw, prepare_splits, engineer_features, build_feature_cols, build_matrices
from integrity   import check_integrity, check_split_leakage
from model       import run_cv, train_final_model
from calibration import fit_calibrator, apply_calibration
pd.set_option('display.max_columns', None)

np.random.seed(SEED)
print('Imports OK')

Imports OK


## 1. Load data

In [2]:
train_raw, val_raw, test = load_raw()

# train: drop missing default_flag (declined/immature) upfront
# val_all: all rows → submission | val_labeled: known outcomes → calibration & optimisation
train, val_all, val_labeled, val_labeled_positions = prepare_splits(train_raw, val_raw)

print(f'train (labeled) : {len(train):>6,}  default rate {train["default_flag"].mean():.4f}')
print(f'val_all         : {len(val_all):>6,}  → submission')
print(f'val_labeled     : {len(val_labeled):>6,}  → calibration / optimisation')
print(f'test            : {len(test):>6,}')

train (labeled) : 51,722  default rate 0.1745
val_all         :  4,489  → submission
val_labeled     :  2,551  → calibration / optimisation
test            :  8,817


## 2. Integrity checks (known planted violations)

In [3]:
check_integrity(train, 'train (labeled)')
print()
check_split_leakage(train, val_all, test)

--- Integrity: train (labeled) ---
  prior_loans_default_count > prior_loans_count: 0
  days_to_default out of [1, 90]: 0
  default_flag / repayment_status mismatch: 0

--- Split leakage (business_id) ---
  train ∩ val  : 0  (expected 0)
  train ∩ test : 0  (expected 0)
  val   ∩ test : 0  (expected 0)


## 3. Feature engineering

In [4]:
train_fe       = engineer_features(train)
val_all_fe     = engineer_features(val_all)
val_labeled_fe = engineer_features(val_labeled)
test_fe        = engineer_features(test)

feature_cols, cat_indices = build_feature_cols(train_fe)
print(f'{len(feature_cols)} features,  {len(cat_indices)} categorical: {[feature_cols[i] for i in cat_indices]}')

50 features,  6 categorical: ['sector', 'geography_region', 'employee_count_bucket', 'intended_use_of_funds', 'owner_personal_credit_band', 'application_channel']


## 4. Prepare feature matrices

`train` is already filtered to labeled rows only (NaN `default_flag` dropped in `prepare_splits`).

`val_labeled_fe` is used for isotonic calibration and conformal coverage checks — it is
completely independent of training (no leakage).  `val_all_fe` (all 4,489 val rows including
unlabeled) feeds the submission predictions.

**Reject inference note:** The model is trained only on approved + matured loans.
Applicants the prior lender declined have no observed outcome, so the model may
under-estimate PD for profiles that were historically rejected. This bias is partially
mitigated by including `prior_underwriter_score` and `has_prior_approval` as features,
and by calibrating on the independent `val_labeled` set.

In [5]:
arrs = build_matrices(train_fe, val_all_fe, val_labeled_fe, test_fe, feature_cols)

print(f'X_train       : {arrs["X_train"].shape}')
print(f'X_val_labeled : {arrs["X_val_labeled"].shape}  (y_val_labeled default rate {arrs["y_val_labeled"].mean():.4f})')
print(f'X_val_all     : {arrs["X_val_all"].shape}')
print(f'X_test        : {arrs["X_test"].shape}')

X_train       : (51722, 50)
X_val_labeled : (2551, 50)  (y_val_labeled default rate 0.2062)
X_val_all     : (4489, 50)
X_test        : (8817, 50)


## 5. 10-fold stratified CatBoost CV *(diagnostic only)*

This step estimates model quality via out-of-fold (OOF) AUC. It does **not** produce the
submission predictions — those come from the ensemble final model in Step 6.

Running CV here lets us detect overfitting (large gap between fold AUC and OOF AUC)
and compare against future experiments without touching the submission path.

In [6]:
# Diagnostic — OOF AUC only. Submission predictions come from train_final_model.
oof_preds, val_preds_folds, test_preds_folds, models = run_cv(
    arrs['X_train'], arrs['y_train'],
    arrs['X_val_all'], arrs['X_test'],
    cat_indices,
)

  Fold  1/10  AUC=0.7736  trees=179
  Fold  2/10  AUC=0.7726  trees=193
  Fold  3/10  AUC=0.7711  trees=103
  Fold  4/10  AUC=0.7744  trees=206
  Fold  5/10  AUC=0.7882  trees=202
  Fold  6/10  AUC=0.7691  trees=289
  Fold  7/10  AUC=0.7835  trees=213
  Fold  8/10  AUC=0.7730  trees=188
  Fold  9/10  AUC=0.7729  trees=288
  Fold 10/10  AUC=0.7828  trees=128

  OOF AUC  : 0.7760
  Fold mean: 0.7761 ± 0.0060


## 6. Ensemble final model — predictions, isotonic calibration, conformal intervals

**Ensemble:** `train_final_model(ensemble=True, n_folds=10)` trains 10 members on different
9/10 subsets (stratified K-fold), each early-stopped on its held-out fold.
`predict_proba()` averages across all 10 members.

**Calibration:** Isotonic regression fitted on `val_labeled` maps raw ensemble scores to
true probabilities. `val_labeled` is completely independent of training — no leakage.

**Prediction intervals (split conformal):**
`q_90 = quantile(|y − p̂|, corrected-0.90)` computed on `val_labeled` residuals.
Interval: `[max(0, p − q_90), min(1, p + q_90)]`.
Guarantees ≥ 90 % marginal coverage under exchangeability (Papadopoulos et al.).

*Note: bootstrap intervals were tried but discarded — on a 51K training set, model
variance across bootstrap resamples is negligible, so bootstrap bands were nearly
zero-width and required a large conformal expansion to reach 90 % coverage, making
them no better than the conformal approach used here.*

In [7]:
final_model = train_final_model(
    arrs['X_train'], arrs['y_train'], cat_indices,
    ensemble=True, n_folds=10,
)
print()

# Raw ensemble predictions
train_raw   = final_model.predict_proba(arrs['X_train'])[:, 1]
val_raw_all = final_model.predict_proba(arrs['X_val_all'])[:, 1]
test_raw    = final_model.predict_proba(arrs['X_test'])[:, 1]

# Isotonic calibration: fit on labeled val, apply to train, val_all, and test
iso = fit_calibrator(val_raw_all, arrs['y_val_labeled'], val_labeled_positions)
val_cal_all, val_cal_labeled, test_cal = apply_calibration(
    iso, val_raw_all, test_raw, val_labeled_positions
)
train_cal = iso.predict(train_raw)

  Fold  1/10  train=46,549  val=5,173  trees=179
  Fold  2/10  train=46,549  val=5,173  trees=156
  Fold  3/10  train=46,550  val=5,172  trees=133
  Fold  4/10  train=46,550  val=5,172  trees=155
  Fold  5/10  train=46,550  val=5,172  trees=240
  Fold  6/10  train=46,550  val=5,172  trees=97
  Fold  7/10  train=46,550  val=5,172  trees=167
  Fold  8/10  train=46,550  val=5,172  trees=201
  Fold  9/10  train=46,550  val=5,172  trees=168
  Fold 10/10  train=46,550  val=5,172  trees=196
Ensemble: 10 members, 97–240 trees (mean 169)



## 7. Prepare survival data for RSF

Target: `days_to_default` (time from origination until observed default).  
Loans with `default_flag = 0` never defaulted — `days_to_default` is null.
They are treated as **right-censored** at the end of the loan term (60 days).

| Column | Meaning |
|--------|---------|
| `duration` | `days_to_default` if defaulted; `LOAN_TERM_DAYS` if censored |
| `event` | 1 = default observed, 0 = censored (repaid or still active) |

In [34]:
LOAN_TERM_DAYS = 60  # right-censoring time for non-defaulted loans

surv_duration = train['days_to_default'].fillna(LOAN_TERM_DAYS).clip(upper=LOAN_TERM_DAYS).astype(float)
surv_event    = train['default_flag'].astype(int)

print(f'Survival training set : {len(surv_duration):,} rows')
print(f'  Events (defaults)   : {surv_event.sum():,}  ({surv_event.mean():.4f})')
print(f'  Censored            : {(surv_event == 0).sum():,}')
print(f'  Duration stats (default): mean={surv_duration[surv_event==1].mean():.1f}  '
      f'median={surv_duration[surv_event==1].median():.1f}  '
      f'range=[{surv_duration[surv_event==1].min():.0f}, {surv_duration[surv_event==1].max():.0f}]')
print(f'  Duration stats (recovered): mean={surv_duration[surv_event==0].mean():.1f}  '
      f'median={surv_duration[surv_event==0].median():.1f}  '
      f'range=[{surv_duration[surv_event==0].min():.0f}, {surv_duration[surv_event==0].max():.0f}]')

Survival training set : 51,722 rows
  Events (defaults)   : 9,024  (0.1745)
  Censored            : 42,698
  Duration stats (default): mean=36.3  median=37.0  range=[3, 60]
  Duration stats (recovered): mean=60.0  median=60.0  range=[60, 60]


## 8. One-hot encode and impute features for RSF

RSF (via scikit-survival) requires fully numeric input and does not support NaN.

- **Categorical features** — one-hot encoded with `drop='first'`; encoder fitted on `train_fe` only.
- **Numeric features** — median imputed (`SimpleImputer`); imputer fitted on `train_fe` only.

No scaling or collinearity removal needed — RSF is scale-invariant and handles correlated features.

In [35]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

cat_feature_names = [feature_cols[i] for i in cat_indices]
num_feature_names = [c for c in feature_cols if c not in cat_feature_names]

ohe     = OneHotEncoder(sparse_output=False, handle_unknown='ignore', drop='first')
imputer = SimpleImputer(strategy='median')

def make_surv_matrix(df_fe, fit=False):
    num = df_fe[num_feature_names].values.astype(float)
    cat = df_fe[cat_feature_names].astype(str)
    if fit:
        num_enc = imputer.fit_transform(num)
        cat_enc = ohe.fit_transform(cat)
    else:
        num_enc = imputer.transform(num)
        cat_enc = ohe.transform(cat)
    return np.hstack([num_enc, cat_enc])

X_surv_train = make_surv_matrix(train_fe, fit=True)
X_surv_val   = make_surv_matrix(val_all_fe)
X_surv_test  = make_surv_matrix(test_fe)

ohe_col_names      = ohe.get_feature_names_out(cat_feature_names).tolist()
surv_feature_names = num_feature_names + ohe_col_names

print(f'Survival feature matrix: {X_surv_train.shape[1]} columns  '
      f'({len(num_feature_names)} numeric + {len(ohe_col_names)} one-hot)')

Survival feature matrix: 63 columns  (44 numeric + 19 one-hot)


## 9. Fit Random Survival Forest

`RandomSurvivalForest` (scikit-survival) extends the standard Random Forest to right-censored
survival data. Each tree is grown on a bootstrap sample using the log-rank split criterion.
The ensemble aggregates cumulative hazard estimates across all trees.

**Imbalance problem:** 82.5 % of training rows are right-censored at exactly day 60.
A bootstrap sample therefore contains ~5 censored observations for every default.
The log-rank criterion then sees overwhelming "survival" signal and learns survival curves
that barely drop below S = 1 over the 60-day term — i.e. nearly everyone is predicted to
survive to day 60, regardless of risk profile.

**Fix — balanced sample weights:**
Each defaulted row is assigned weight `n_total / (2 × n_events) ≈ 2.87` and each
censored row weight `n_total / (2 × n_censored) ≈ 0.61`, so both groups contribute
equal total weight to every log-rank split.  This forces the RSF to learn default-hazard
timing across the full 1–60-day range rather than collapsing everything to the end.

`n_estimators=200` (up from 100) compensates for the slightly higher tree variance
that comes with weighted bootstrap samples.

In [36]:
from sksurv.ensemble import RandomSurvivalForest
from sksurv.util import Surv

y_surv = Surv.from_arrays(event=surv_event.astype(bool), time=surv_duration)

# Balanced sample weights: upweight defaults so each group contributes equally
# to the log-rank split criterion, forcing the RSF to learn hazard timing
# across the full 1-60 day range rather than clustering at day 60.
n_total    = len(surv_event)
n_events   = int(surv_event.sum())
n_censored = n_total - n_events

sw = np.where(
    surv_event.astype(bool),
    n_total / (2 * n_events),    # ~2.87x for defaulted rows
    n_total / (2 * n_censored),  # ~0.61x for censored rows
)
print(f'Events: {n_events:,}  Censored: {n_censored:,}  '
      f'(default rate {n_events/n_total:.4f})')
print(f'Sample weights — defaulted: {sw[surv_event.astype(bool)][0]:.4f}  '
      f'censored: {sw[~surv_event.astype(bool)][0]:.4f}')

rsf = RandomSurvivalForest(
    n_estimators=200,
    min_samples_leaf=15,
    max_features='sqrt',
    random_state=SEED,
    n_jobs=-1,
)
rsf.fit(X_surv_train, y_surv, sample_weight=sw)
print(f'Concordance index (train): {rsf.score(X_surv_train, y_surv):.4f}')

Events: 9,024  Censored: 42,698  (default rate 0.1745)
Sample weights — defaulted: 2.8658  censored: 0.6057
Concordance index (train): 0.8569


## 10. Conditional expected default day — E[T | default occurs]

The RSF training set is 82.5% right-censored (non-defaulting loans censored at 60 days).
This heavy imbalance biases S(t) upward — for ~75% of applicants S(t) never drops to 0.5,
so the **median** survival time is 60 for most of the portfolio.

**Why the median is wrong for the NPV threshold:**

The daily-draw structure means defaults at day > ~52 are actually *profitable*
(enough principal has been collected), so `NPV_default(d=60)` is positive (+$1,599 on a $10k loan).
Using median = 60 therefore produces either:
- `NPV_default > 0` → threshold < 0 → clip to 0 → decline everyone (wrong), or
- The old patch `npv_d = 0` → threshold = 1.0 → approve 94%+ (wrong)

**Correct question to ask the RSF:** not "when will this person survive to?"
but **"if this person does default, when will it happen?"**

This is the conditional expected default day E[T | T ≤ 60]:

```
E[T | T ≤ 60] = Σ_{d=1}^{60}  d · [S(d−1) − S(d)] / [1 − S(60)]
```

For applicants where the RSF assigns near-zero hazard (S(60) ≈ 1.0), we fall back to
the empirical training mean of 36 days — the best prior on default timing we have.
This conditional day is always in (1, 60], is decoupled from survivorship bias in S(60),
and feeds a well-behaved NPV_default that represents genuine early-default loss.

In [37]:
EMPIRICAL_MEAN_DEFAULT_DAY = 36.0  # mean days_to_default for defaulted loans in training

def compute_conditional_default_day(X, loan_term=LOAN_TERM_DAYS,
                                    fallback=EMPIRICAL_MEAN_DEFAULT_DAY):
    """
    E[T | T <= loan_term] from RSF survival functions.

    Uses searchsorted for efficiency instead of a per-day loop.
    Falls back to the empirical training mean (36 days) when the RSF
    assigns near-zero hazard — prevents survivorship bias from collapsing
    the conditional distribution.
    """
    surv_funcs = rsf.predict_survival_function(X)
    eval_days  = np.arange(1, loan_term + 1, dtype=float)
    result = []

    for sf in surv_funcs:
        # S(d) for each integer day: value of the step function at the last
        # knot at or before d.  searchsorted(..., 'right') - 1 gives that index.
        idx      = np.searchsorted(sf.x, eval_days, side='right') - 1
        s_at_day = np.where(idx >= 0, sf.y[np.maximum(idx, 0)], 1.0)

        # P(T = d) = S(d-1) - S(d); prepend S(0) = 1
        s_prev  = np.concatenate([[1.0], s_at_day[:-1]])
        density = np.clip(s_prev - s_at_day, 0.0, None)  # numerical safety
        total   = density.sum()                           # ≈ P(T ≤ loan_term)

        if total < 1e-6:
            result.append(fallback)
        else:
            result.append(float(np.dot(eval_days, density) / total))

    return pd.Series(result).reset_index(drop=True)


predicted_days_to_default_train = compute_conditional_default_day(X_surv_train)
val_days_to_default             = compute_conditional_default_day(X_surv_val)
test_days_to_default            = compute_conditional_default_day(X_surv_test)

print('Conditional expected default day  E[T | default occurs, T ≤ 60]')
print(f'Train ({len(predicted_days_to_default_train):,}) : '
      f'mean={predicted_days_to_default_train.mean():.1f}  '
      f'median={predicted_days_to_default_train.median():.1f}  '
      f'range=[{predicted_days_to_default_train.min():.1f}, {predicted_days_to_default_train.max():.1f}]')
print(f'Val   ({len(val_days_to_default):,}) : '
      f'mean={val_days_to_default.mean():.1f}  '
      f'median={val_days_to_default.median():.1f}  '
      f'range=[{val_days_to_default.min():.1f}, {val_days_to_default.max():.1f}]')
print(f'Test  ({len(test_days_to_default):,}) : '
      f'mean={test_days_to_default.mean():.1f}  '
      f'median={test_days_to_default.median():.1f}  '
      f'range=[{test_days_to_default.min():.1f}, {test_days_to_default.max():.1f}]')
print()

predicted_days_to_default = pd.concat([val_days_to_default, test_days_to_default], ignore_index=True)
predicted_days_to_default.name = 'predicted_days_to_default'
print(predicted_days_to_default.describe().round(2))

Conditional expected default day  E[T | default occurs, T ≤ 60]
Train (51,722) : mean=42.0  median=41.9  range=[15.0, 59.7]
Val   (4,489) : mean=39.5  median=39.4  range=[18.8, 58.6]
Test  (8,817) : mean=39.4  median=39.4  range=[17.7, 59.2]

count    13306.00
mean        39.45
std          5.99
min         17.70
25%         35.56
50%         39.40
75%         43.31
max         59.20
Name: predicted_days_to_default, dtype: float64


## 11. Predict recovery rate — CatBoost regressor

`recovery_pct = final_recovered_amount / requested_amount` is computed for every defaulted loan
in training and clipped to [0, 1].

A `CatBoostRegressor` is trained on those loans only, using the same 47 features and
categorical indices as the default-probability model. 20 % of defaulted loans are held out
for RMSE / MAE evaluation. Predictions for all val + test applicants give
`predicted_recovery_pct` — "if this loan defaults, what fraction is recovered."

In [38]:
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split

defaulted_mask = train['default_flag'].astype(bool)
defaulted      = train[defaulted_mask].copy()
defaulted['recovery_pct'] = (
    defaulted['final_recovered_amount'] / defaulted['requested_amount'].clip(lower=1)
).clip(0, 1)

y_rec = defaulted['recovery_pct'].values
X_rec = arrs['X_train'][defaulted_mask.values]

print('── Recovery % on defaulted training loans ────────────────────')
print(defaulted['recovery_pct'].describe().round(4))
print()

# 80 / 20 split for hold-out evaluation
X_tr, X_hout, y_tr, y_hout = train_test_split(X_rec, y_rec, test_size=0.2, random_state=SEED)

rec_model = CatBoostRegressor(
    iterations=1000, learning_rate=0.05, depth=6,
    loss_function='RMSE', cat_features=cat_indices,
    random_seed=SEED, verbose=0,
)
rec_model.fit(X_tr, y_tr, eval_set=(X_hout, y_hout), early_stopping_rounds=50)

hout_pred = rec_model.predict(X_hout).clip(0, 1)
rmse = np.sqrt(np.mean((hout_pred - y_hout) ** 2))
mae  = np.mean(np.abs(hout_pred - y_hout))
print(f'Hold-out RMSE : {rmse:.4f}')
print(f'Hold-out MAE  : {mae:.4f}')
print()

# Predict for every val_all + test applicant
predicted_recovery_pct = pd.Series(
    np.clip(
        np.concatenate([
            rec_model.predict(arrs['X_val_all']),
            rec_model.predict(arrs['X_test']),
        ]),
        0, 1,
    ),
    name='predicted_recovery_pct',
)

print('── Predicted recovery % — val + test ────────────────────────')
print(predicted_recovery_pct.describe().round(4))

── Recovery % on defaulted training loans ────────────────────
count    9024.0000
mean        0.0914
std         0.0903
min         0.0000
25%         0.0046
50%         0.0715
75%         0.1476
max         0.6205
Name: recovery_pct, dtype: float64

Hold-out RMSE : 0.0900
Hold-out MAE  : 0.0732

── Predicted recovery % — val + test ────────────────────────
count    13306.0000
mean         0.0847
std          0.0173
min          0.0000
25%          0.0753
50%          0.0843
75%          0.0946
max          0.1639
Name: predicted_recovery_pct, dtype: float64


## 12. Per-loan break-even PD threshold

For each applicant the break-even threshold is the probability of default at which
expected NPV = 0:

```
threshold = NPV_repaid / (NPV_repaid − NPV_default)
```

- `default_day` = **conditional expected default day** E[T | default] from Step 10
  (not the biased median — see Step 10 for why)
- `recovered_amt` = `predicted_recovery_pct × requested_amount`

**Handling late defaults (NPV_default > NPV_repaid):**
When the conditional default day is > ~52 days, `NPV_default > NPV_repaid`
(enough daily draws collected to be more profitable than full repayment).
In this case the denominator `NPV_repaid − NPV_default` is negative, giving
a threshold < 0. The correct interpretation is "approve regardless of PD" → threshold = 1.
This is handled explicitly before clipping.

In [39]:
def calculate_npv_repaid(requested_amt, fee_ratio = 0.03, r = 0.35, T = 60):
    F = fee_ratio*requested_amt
    return F + ((requested_amt*r*T)/365)

def calculate_npv_default(requested_amt, default_day, recovered_amt, fee_ratio = 0.03, r = 0.35, T = 60):
    F = fee_ratio*requested_amt
    daily_draw = requested_amt * (1 + (r*T/365))/T
    return F + daily_draw*(default_day - 1) + recovered_amt - requested_amt

def threshold(npv_repaid, npv_default):
    return npv_repaid/(npv_repaid - npv_default)

In [40]:
# Recovery-rate predictions for training rows
train_recovery_pred = pd.Series(
    np.clip(rec_model.predict(arrs['X_train']), 0, 1)
).reset_index(drop=True)

def compute_thresholds(requested_amts, default_days, recovery_pcts):
    req   = requested_amts.values
    rec   = recovery_pcts.values * req
    npv_r = calculate_npv_repaid(req)
    npv_d = calculate_npv_default(req, default_days.values, rec)
    denom = npv_r - npv_d
    # When denom <= 0, NPV_default >= NPV_repaid — late default is at least as
    # profitable as repayment, so approve regardless of PD (threshold = 1).
    # Otherwise clip the computed threshold to [0, 1].
    t = np.where(denom <= 0, 1.0, np.clip(npv_r / denom, 0.0, 1.0))
    return pd.Series(t)

# Train
train_thresholds = compute_thresholds(
    train['requested_amount'].reset_index(drop=True),
    predicted_days_to_default_train,
    train_recovery_pred,
)

# Val + Test
all_applicants      = pd.concat([val_all, test], ignore_index=True)
all_requested       = all_applicants['requested_amount'].reset_index(drop=True)
val_test_thresholds = compute_thresholds(all_requested, predicted_days_to_default, predicted_recovery_pct)

val_thresholds  = val_test_thresholds.iloc[:len(val_all)].reset_index(drop=True)
test_thresholds = val_test_thresholds.iloc[len(val_all):].reset_index(drop=True)

print('Per-loan break-even PD threshold  (using conditional expected default day)')
print(f'Train ({len(train_thresholds):>6,}) : mean={train_thresholds.mean():.4f}  '
      f'std={train_thresholds.std():.4f}  '
      f'range=[{train_thresholds.min():.4f}, {train_thresholds.max():.4f}]')
print(f'Val   ({len(val_thresholds):>6,}) : mean={val_thresholds.mean():.4f}  '
      f'std={val_thresholds.std():.4f}  '
      f'range=[{val_thresholds.min():.4f}, {val_thresholds.max():.4f}]')
print(f'Test  ({len(test_thresholds):>6,}) : mean={test_thresholds.mean():.4f}  '
      f'std={test_thresholds.std():.4f}  '
      f'range=[{test_thresholds.min():.4f}, {test_thresholds.max():.4f}]')
print()
for label, t in [('Val', val_thresholds), ('Test', test_thresholds)]:
    print(f'{label} threshold distribution:')
    print(f'  < 0.25  : {(t < 0.25).sum():>5,}  ({(t < 0.25).mean():.1%})  ← decline if PD ≥ 25%')
    print(f'  0.25–0.5: {((t >= 0.25) & (t < 0.5)).sum():>5,}  ({((t >= 0.25) & (t < 0.5)).mean():.1%})')
    print(f'  0.5–0.9 : {((t >= 0.5)  & (t < 0.9)).sum():>5,}  ({((t >= 0.5) & (t < 0.9)).mean():.1%})')
    print(f'  ≥ 0.9   : {(t >= 0.9).sum():>5,}  ({(t >= 0.9).mean():.1%})  ← very lenient')

Per-loan break-even PD threshold  (using conditional expected default day)
Train (51,722) : mean=0.4048  std=0.1922  range=[0.1257, 1.0000]
Val   ( 4,489) : mean=0.3375  std=0.1468  range=[0.1401, 1.0000]
Test  ( 8,817) : mean=0.3389  std=0.1519  range=[0.1344, 1.0000]

Val threshold distribution:
  < 0.25  : 1,236  (27.5%)  ← decline if PD ≥ 25%
  0.25–0.5: 2,800  (62.4%)
  0.5–0.9 :   365  (8.1%)
  ≥ 0.9   :    88  (2.0%)  ← very lenient
Test threshold distribution:
  < 0.25  : 2,459  (27.9%)  ← decline if PD ≥ 25%
  0.25–0.5: 5,472  (62.1%)
  0.5–0.9 :   701  (8.0%)
  ≥ 0.9   :   185  (2.1%)  ← very lenient


In [41]:
#deny
train_cal>=train_thresholds
val_cal_all>=val_thresholds
test_cal>=test_thresholds

0       False
1       False
2        True
3       False
4       False
        ...  
8812     True
8813    False
8814    False
8815    False
8816     True
Length: 8817, dtype: bool

## 13. Build submission file A

**Decision:** 1 = approve (`predicted_pd < threshold`), 0 = decline otherwise.

**90% confidence interval on `predicted_pd`:**
The 10 CV fold models (Step 5) were each trained on a different 9/10 data subset.
Applying the same isotonic calibrator to each fold's raw predictions gives 10 calibrated
PD estimates per applicant. Their standard deviation measures model estimation uncertainty —
tight when all 10 models agree, wider when they disagree.

`CI = predicted_pd ± 1.645 × std(calibrated fold predictions)`  (z-score for 90% two-sided)

This is a confidence interval on the PD *estimate*, not a prediction interval on the outcome.

In [42]:
Z_90 = 1.645  # z-score for 90% two-sided CI

# val_preds_folds shape: (n_val_all, n_folds) — each column is one fold model's raw predictions.
# Apply the same iso calibrator to each column, then stack into (n_folds, n_val_all).
cal_val_folds  = np.vstack([iso.predict(val_preds_folds[:,  k]) for k in range(val_preds_folds.shape[1])])
cal_test_folds = np.vstack([iso.predict(test_preds_folds[:, k]) for k in range(test_preds_folds.shape[1])])

val_se  = cal_val_folds.std(axis=0)   # (n_val_all,)
test_se = cal_test_folds.std(axis=0)  # (n_test,)

val_lower  = np.clip(val_cal_all - Z_90 * val_se,  0.0, 1.0)
val_upper  = np.clip(val_cal_all + Z_90 * val_se,  0.0, 1.0)
test_lower = np.clip(test_cal    - Z_90 * test_se, 0.0, 1.0)
test_upper = np.clip(test_cal    + Z_90 * test_se, 0.0, 1.0)

print(f'90% CI half-width  val : mean={np.mean(Z_90 * val_se):.4f}  '
      f'median={np.median(Z_90 * val_se):.4f}')
print(f'90% CI half-width test : mean={np.mean(Z_90 * test_se):.4f}  '
      f'median={np.median(Z_90 * test_se):.4f}')
print()

# Re-enforce predicted_pd within bounds after boundary clipping
val_lower  = np.minimum(val_lower,  val_cal_all)
val_upper  = np.maximum(val_upper,  val_cal_all)
test_lower = np.minimum(test_lower, test_cal)
test_upper = np.maximum(test_upper, test_cal)

assert np.all((val_cal_all >= val_lower)  & (val_cal_all <= val_upper)),  'val PD outside CI'
assert np.all((test_cal    >= test_lower) & (test_cal    <= test_upper)), 'test PD outside CI'
print('Assertion passed: pd_lower_90 <= predicted_pd <= pd_upper_90 for all rows')
print()

# Decision: 1 = approve (pd < threshold), 0 = decline (pd >= threshold)
val_decision  = (val_cal_all  < val_thresholds.values).astype(int)
test_decision = (test_cal     < test_thresholds.values).astype(int)

# Assemble submission (required columns only)
val_sub = pd.DataFrame({
    'applicant_id': val_all['applicant_id'].values,
    'decision':     val_decision,
    'predicted_pd': np.round(val_cal_all, 6),
    'pd_lower_90':  np.round(val_lower,   6),
    'pd_upper_90':  np.round(val_upper,   6),
})
test_sub = pd.DataFrame({
    'applicant_id': test['applicant_id'].values,
    'decision':     test_decision,
    'predicted_pd': np.round(test_cal,    6),
    'pd_lower_90':  np.round(test_lower,  6),
    'pd_upper_90':  np.round(test_upper,  6),
})

submission = pd.concat([val_sub, test_sub], ignore_index=True)

SUB_DIR.mkdir(exist_ok=True)
out_path = SUB_DIR / 'submission_A.csv'
submission.to_csv(out_path, index=False)

print(f'Saved {len(submission):,} rows  →  {out_path}')
print()
print(f'val  ({len(val_sub):,}) :  approve={val_decision.sum():,}  '
      f'decline={(val_decision==0).sum():,}  '
      f'(decline rate {(val_decision==0).mean():.1%})')
print(f'test ({len(test_sub):,}) :  approve={test_decision.sum():,}  '
      f'decline={(test_decision==0).sum():,}  '
      f'(decline rate {(test_decision==0).mean():.1%})')
print()
print(submission.head(3).to_string(index=False))

90% CI half-width  val : mean=0.0533  median=0.0326
90% CI half-width test : mean=0.0535  median=0.0327

Assertion passed: pd_lower_90 <= predicted_pd <= pd_upper_90 for all rows

Saved 13,306 rows  →  submissions/submission_A.csv

val  (4,489) :  approve=2,890  decline=1,599  (decline rate 35.6%)
test (8,817) :  approve=5,656  decline=3,161  (decline rate 35.9%)

                        applicant_id  decision  predicted_pd  pd_lower_90  pd_upper_90
741ca337-bb2e-e972-e628-1d9a1bbc226f         0      0.308411     0.131975     0.484848
d1f34701-2a39-a190-7b54-059e6cbdae38         0      0.682540     0.533210     0.831870
0fb6a398-0e87-6f6f-92f9-836af0d3bf3f         0      1.000000     1.000000     1.000000


## 14. Deliverable B — Default Trajectory Forecast

For each origination cohort week (1–13) and loan age (1–13 weeks), predict the cumulative
fraction of **approved** cohort loans that have defaulted by day `7 × loan_age_weeks`.

**Approach:**
1. Re-fit RSF with 91-day censoring (13 weeks, uncapped) so the survival function spans the full horizon.
2. For each approved val/test loan, evaluate `CDR_i(t) = 1 − S_i(t)` at t = 7, 14, …, 91 days.
3. Aggregate: `CDR_cohort(t) = mean(CDR_i(t))` across approved loans in that cohort.
4. 90% CI: `CDR ± 1.645 × std(CDR_i) / √n` (SE of the sample mean).
5. Enforce monotonicity within each cohort via cumulative max.

# --- 14a. Re-fit RSF with 91-day (13-week) censoring ---

In [43]:
from pathlib import Path

LOAN_TERM_B = 91  # 13 weeks × 7 days; raw data max days_to_default = 90 < 91

surv_duration_b = train['days_to_default'].fillna(LOAN_TERM_B).astype(float)
y_surv_b        = Surv.from_arrays(event=surv_event.astype(bool), time=surv_duration_b)

# Reuse the same balanced weights from Step 9 — censoring rate is identical,
# only the censoring time changes from 60 → 91.
rsf_b = RandomSurvivalForest(
    n_estimators=200, min_samples_leaf=15, max_features='sqrt',
    random_state=SEED, n_jobs=-1,
)
rsf_b.fit(X_surv_train, y_surv_b, sample_weight=sw)
print(f'RSF_B concordance (train): {rsf_b.score(X_surv_train, y_surv_b):.4f}')
print(f'Duration range: [{surv_duration_b.min():.0f}, {surv_duration_b.max():.0f}] days')
print()

RSF_B concordance (train): 0.8569
Duration range: [3, 91] days



# --- 14b. Assign approved val/test loans to cohort weeks ---

In [44]:
COHORT_DEFS = pd.read_csv(Path('dataset') / 'cohort_week_definitions.csv',
                          parse_dates=['start_date', 'end_date'])

def assign_cohort_week(ts_series):
    dates  = pd.to_datetime(ts_series).dt.date
    result = pd.Series(-1, index=ts_series.index, name='cohort_week')
    for _, row in COHORT_DEFS.iterrows():
        mask = (dates >= row['start_date'].date()) & (dates <= row['end_date'].date())
        result[mask] = int(row['cohort_week'])
    return result

val_approved_mask  = (val_decision  == 1)
test_approved_mask = (test_decision == 1)

X_val_approved  = X_surv_val[val_approved_mask]
X_test_approved = X_surv_test[test_approved_mask]

val_cohorts  = assign_cohort_week(val_all['application_timestamp'])[val_approved_mask].reset_index(drop=True)
test_cohorts = assign_cohort_week(test['application_timestamp'])[test_approved_mask].reset_index(drop=True)

all_cohorts = pd.concat([val_cohorts, test_cohorts], ignore_index=True)
counts = all_cohorts.value_counts().sort_index()
print('Approved loans per cohort week:')
for cw, n in counts.items():
    print(f'  week {cw:2d}: {n:,}')
print(f'  Total: {len(all_cohorts):,}  (unassigned: {(all_cohorts == -1).sum()})')
print()

Approved loans per cohort week:
  week  1: 708
  week  2: 716
  week  3: 695
  week  4: 715
  week  5: 727
  week  6: 684
  week  7: 675
  week  8: 694
  week  9: 630
  week 10: 646
  week 11: 527
  week 12: 538
  week 13: 591
  Total: 8,546  (unassigned: 0)



# --- 14c. Compute CDR matrix for every approved loan ---

In [45]:
LOAN_AGE_WEEKS = list(range(1, 14))
EVAL_DAYS      = [7 * w for w in LOAN_AGE_WEEKS]  # 7, 14, ..., 91

def get_cdr_matrix(X, times):
    """Returns (n_loans, len(times)) array of CDR(t) = 1 - S(t)."""
    surv_funcs = rsf_b.predict_survival_function(X)
    cdr = np.zeros((len(surv_funcs), len(times)))
    for i, sf in enumerate(surv_funcs):
        for j, t in enumerate(times):
            mask = sf.x <= t
            cdr[i, j] = 1.0 - (float(sf.y[mask][-1]) if mask.any() else 1.0)
    return cdr

print('Computing CDR matrix for approved val loans ...')
cdr_val_appr  = get_cdr_matrix(X_val_approved,  EVAL_DAYS)
print('Computing CDR matrix for approved test loans ...')
cdr_test_appr = get_cdr_matrix(X_test_approved, EVAL_DAYS)
print('Done.')

Computing CDR matrix for approved val loans ...
Computing CDR matrix for approved test loans ...
Done.


# --- 14d. Load template, fill prediction columns, save ---

In [46]:
submission_b = pd.read_csv(Path('dataset') / 'submission_B_template.csv')

for cw in range(1, 14):
    val_idx  = np.where(val_cohorts.values  == cw)[0]
    test_idx = np.where(test_cohorts.values == cw)[0]

    parts = []
    if len(val_idx)  > 0: parts.append(cdr_val_appr[val_idx])
    if len(test_idx) > 0: parts.append(cdr_test_appr[test_idx])

    if len(parts) == 0:
        cdrs = lowers = uppers = np.zeros(13)
    else:
        cdr_cohort = np.vstack(parts)          # (n_cohort, 13)
        n          = len(cdr_cohort)
        cdrs   = cdr_cohort.mean(axis=0)
        ses    = cdr_cohort.std(axis=0) / np.sqrt(n)
        lowers = np.clip(cdrs - Z_90 * ses, 0.0, 1.0)
        uppers = np.clip(cdrs + Z_90 * ses, 0.0, 1.0)

    # Enforce monotonicity (cumulative rates never decrease)
    cdrs   = np.maximum.accumulate(cdrs)
    lowers = np.maximum.accumulate(lowers)
    uppers = np.maximum.accumulate(uppers)

    # Ensure lower <= CDR <= upper after monotonicity pass
    lowers = np.minimum(lowers, cdrs)
    uppers = np.maximum(uppers, cdrs)

    mask = submission_b['cohort_week'] == cw
    submission_b.loc[mask, 'cumulative_default_rate'] = np.round(cdrs,   6)
    submission_b.loc[mask, 'cdr_lower_90']            = np.round(lowers, 6)
    submission_b.loc[mask, 'cdr_upper_90']            = np.round(uppers, 6)

# Validation checks
assert len(submission_b) == 169, 'Wrong number of rows'
assert (submission_b['cdr_lower_90'] <= submission_b['cumulative_default_rate']).all(), \
    'lower > CDR'
assert (submission_b['cumulative_default_rate'] <= submission_b['cdr_upper_90']).all(), \
    'CDR > upper'
for cw in range(1, 14):
    cdr_seq = submission_b[submission_b['cohort_week'] == cw].sort_values('loan_age_weeks')
    assert (cdr_seq['cumulative_default_rate'].diff().dropna() >= -1e-9).all(), \
        f'CDR not monotone for cohort {cw}'
print('All validation checks passed.')
print()

# Save
out_path_b = SUB_DIR / 'submission_B_trajectory.csv'
submission_b.to_csv(out_path_b, index=False)
print(f'Saved {len(submission_b)} rows  →  {out_path_b}')
print()

# Summary: CDR by cohort_week × loan_age_weeks
summary = submission_b.pivot(index='cohort_week', columns='loan_age_weeks',
                             values='cumulative_default_rate')
print('CDR by cohort_week (rows) × loan_age_weeks (cols):')
print(summary.round(4).to_string())

All validation checks passed.

Saved 169 rows  →  submissions/submission_B_trajectory.csv

CDR by cohort_week (rows) × loan_age_weeks (cols):
loan_age_weeks      1       2       3       4       5       6       7       8       9       10      11      12      13
cohort_week                                                                                                           
1               0.0147  0.0413  0.0631  0.0894  0.1163  0.1404  0.1636  0.1897  0.2012  0.2012  0.2012  0.2012  0.3031
2               0.0145  0.0412  0.0630  0.0896  0.1168  0.1414  0.1646  0.1916  0.2033  0.2033  0.2033  0.2033  0.3054
3               0.0142  0.0396  0.0606  0.0855  0.1112  0.1345  0.1570  0.1823  0.1935  0.1935  0.1935  0.1935  0.2954
4               0.0142  0.0408  0.0624  0.0883  0.1144  0.1385  0.1616  0.1878  0.1994  0.1994  0.1994  0.1994  0.3001
5               0.0146  0.0411  0.0627  0.0885  0.1147  0.1387  0.1619  0.1878  0.1995  0.1995  0.1995  0.1995  0.3002
6               0.0141  0

## 15. Deliverable C — Counterfactual PD predictions

For each query in `intervention_queries.csv`: set one feature to its intervention value,
hold everything else fixed, and predict PD with the same ensemble + isotonic calibrator
used in Part A (no retraining).

**Approach:**
1. Look up the applicant's original raw test row.
2. Set `feature_name = intervention_value` (re-casting to the correct dtype).
3. Re-run `engineer_features` so all derived features (e.g. `prior_default_rate`,
   `loan_to_annual_rev`, MNAR indicators) are recomputed from the modified raw values.
4. Predict with `final_model` → `iso` for the point estimate.
5. Apply all 10 CV fold models and use `1.645 × std` for the 90% CI — identical method to Part A.

In [47]:
from pathlib import Path

queries = pd.read_csv(Path('dataset') / 'intervention_queries.csv')

# All 300 unique applicants are in the test set
test_id_pos = {aid: i for i, aid in enumerate(test['applicant_id'])}

# Categorical features are integer-coded in the raw data
cat_raw_set = set(cat_feature_names)  # sector, geography_region, etc.

# Build one modified raw row per query, then batch through engineer_features
modified_rows = []
for _, q in queries.iterrows():
    row  = test.iloc[test_id_pos[q['applicant_id']]].copy()
    feat = q['feature_name']
    val  = q['intervention_value']
    # Preserve integer dtype for categorical features; float for everything else
    row[feat] = int(val) if feat in cat_raw_set else float(val)
    modified_rows.append(row)

modified_df = pd.DataFrame(modified_rows).reset_index(drop=True)

# Re-derive engineered features (prior_default_rate, loan_to_annual_rev,
# MNAR indicators, etc. all update automatically from the modified raw values)
modified_fe = engineer_features(modified_df)
X_cf = modified_fe[feature_cols].values

# ── Point estimate ────────────────────────────────────────────────────────────
raw_cf = final_model.predict_proba(X_cf)[:, 1]
pd_cf  = iso.predict(raw_cf)

# ── 90% CI via 10 CV fold models (same method as Part A) ────────────────────
cal_cf_folds = np.vstack([iso.predict(m.predict_proba(X_cf)[:, 1]) for m in models])
cf_se        = cal_cf_folds.std(axis=0)

pd_cf_lower = np.clip(pd_cf - Z_90 * cf_se, 0.0, 1.0)
pd_cf_upper = np.clip(pd_cf + Z_90 * cf_se, 0.0, 1.0)
pd_cf_lower = np.minimum(pd_cf_lower, pd_cf)   # ensure lower <= point estimate
pd_cf_upper = np.maximum(pd_cf_upper, pd_cf)   # ensure upper >= point estimate

assert np.all((pd_cf >= pd_cf_lower) & (pd_cf <= pd_cf_upper)), 'CF PD outside CI bounds'
print('Assertion passed: pd_cf_lower_90 <= predicted_pd_cf <= pd_cf_upper_90 for all rows')
print()

submission_c = pd.DataFrame({
    'query_id':        queries['query_id'].values,
    'predicted_pd_cf': np.round(pd_cf,       6),
    'pd_cf_lower_90':  np.round(pd_cf_lower, 6),
    'pd_cf_upper_90':  np.round(pd_cf_upper, 6),
})

SUB_DIR.mkdir(exist_ok=True)
out_path_c = SUB_DIR / 'submission_C_counterfactuals.csv'
submission_c.to_csv(out_path_c, index=False)
print(f'Saved {len(submission_c)} rows  →  {out_path_c}')
print()
print(f'CI half-width : mean={np.mean(Z_90 * cf_se):.4f}  median={np.median(Z_90 * cf_se):.4f}')
print()
print(submission_c.head(6).to_string(index=False))

Assertion passed: pd_cf_lower_90 <= predicted_pd_cf <= pd_cf_upper_90 for all rows

Saved 900 rows  →  submissions/submission_C_counterfactuals.csv

CI half-width : mean=0.0631  median=0.0393

query_id  predicted_pd_cf  pd_cf_lower_90  pd_cf_upper_90
  q00000         0.222222        0.000000        0.446623
  q00001         0.153675        0.091410        0.215940
  q00002         0.222222        0.000000        0.446623
  q00003         0.682540        0.590829        0.774250
  q00004         0.772727        0.586551        0.958903
  q00005         0.682540        0.615698        0.749381
